In [9]:
import os
from dotenv import load_dotenv
load_dotenv()
from util import split_wav_by_timestamps

token = os.environ.get('HUGGINGFACE_ACCESS_TOKEN')
mp3_file = "530.mp3"
wav_file = "530.wav"

In [10]:
from pyannote.audio import Audio, Pipeline
import torch

io = Audio(mono='downmix', sample_rate=16000)
waveform, sample_rate = io(mp3_file)

pipeline = Pipeline.from_pretrained(
    "pyannote/speaker-diarization-3.1",
    use_auth_token=token)

pipeline.to(torch.device("cuda"))

diarization = pipeline({"waveform": waveform, "sample_rate": sample_rate}, num_speakers=2)

In [ ]:
timestamps = []

for turn, _, speaker in diarization.itertracks(yield_label=True):
    timestamps.append((turn.start, turn.end, speaker))

In [17]:
output_dir = 'out'

if not os.path.exists(output_dir):
    os.makedirs(output_dir)

split_wav_by_timestamps(wav_file, output_dir, timestamps)